# Tratativa das estatísticas
## Etapa tratativa dos dados e gravação na silver

Nem todas as partidas possuem a informação de `offsides`, com isso, ao invés de termos 4 métricas para algumas partidas, teremos apenas 3.

Além disso, algumas métricas aparecem duplicadas e as vezes com valores distintos. Mantive os registros com os maiores valores registrados.

Métricas:

    - Corners
    - Fouls
    - Penalty
    - Offsides


In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window
from pyspark.sql.types import *

In [0]:
df_silver = spark.read.table("apifootball.silver.statistics") #não existe na primeira execução
max_dh_bronze = (
    spark.read.table("apifootball.bronze.matches_raw")
    .agg(max("dh_processing_br").alias("max_dh"))
    .collect()[0]["max_dh"]
)
df_bronze = (
    spark.read.table("apifootball.bronze.matches_raw").alias("r")
    .filter(col("r.match_status") == "Finished")
    .filter(col("r.dh_processing_br") == max_dh_bronze)
    .join(df_silver.alias("s"), col("r.match_id") == col("s.match_id"), "leftanti") # segunda rodada +, primeira comentar
)#167

In [0]:
stats_schema = ArrayType(
    StructType([
        StructField("type", StringType()),
        StructField("home", StringType()),
        StructField("away", StringType())
    ])
)

In [0]:
df_stats = (
    df_bronze
    .withColumn(
        "stats_array",
        from_json(col("statistics"), stats_schema)
    )
    .withColumn(
        "stat",
        explode_outer("stats_array")
    )
)

In [0]:
df_match_stats = (
    df_stats
    .filter(
        col("stat.type").isin(
            "Corners",
            "Fouls",
            "Penalty",
            "Offsides"
        )
    )
    .select(
        col("match_id").cast("int").alias("match_id"),
        col("stat.type").alias("statistic"),
        col("match_hometeam_id").cast("int").alias("hometeam_id"),
        col("match_hometeam_name").alias("hometeam_name"),
        col("stat.home").cast("int").alias("home_value"),
        col("match_awayteam_id").cast("int").alias("awayteam_id"),
        col("match_awayteam_name").alias("awayteam_name"),
        col("stat.away").cast("int").alias("away_value"),
        (col("stat.home").cast("int") + col("stat.away").cast("int")).alias("total_value"),
        expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br'),
        col('dh_processing_br').alias('datetime_processing_bronze_br')        
    )
).orderBy("match_id")

In [0]:
# mesmo com distinct, ainda tem alguns arrays duplicados com valores diferentes. Manterei o maior valor.
window_spec = Window.partitionBy("match_id", "statistic").orderBy("total_value")

df_match_stats_rn = df_match_stats.withColumn("rn", row_number().over(window_spec))
df_match_stats_dedup = (
    df_match_stats_rn.filter(col("rn") == 1).drop("rn").orderBy(col("match_id"))
)  # 622

In [0]:
df_match_stats_dedup.write.mode("append").saveAsTable("apifootball.silver.statistics")

In [0]:
#spark.read.table("apifootball.silver.statistics").display()